# 6. Paper figures

The figures for a paper on the forced response: the **main figures** put the models together, and the **supplementary figures** show each model. They are drawn from what notebook 03 saved, with the same colour scales as notebook 04, in print style: 6.5–8 pt text, every figure 183 mm (7.2 in) wide (a two-column page), panel letters, and no titles (the caption above each figure is its title). *Check the colour ranges* says whether the scales suit the data. Each is saved as a PDF (vector) and a 300 dpi PNG in `figures/<variable>/paper/`.

| Figure | What it shows |
|---|---|
| 1 | The distribution, after Bracegirdle et al. (2024, Fig. 1): Q05, the mean, Q95 and the width in hist-nat and historical, and the change in each (multi-model median), in winter (JJA) |
| 2, 3 | How each forcing changes the distribution: the bottom row of Fig. 1 for every forcing, in winter (JJA) and summer (DJF) |
| 4 | Where the mean and the width have changed together, every forcing and both seasons, on one 3 × 3 colour key |
| 5 | Area means south of 60°S: every model's change in the mean and in the width, every forcing and season |
| 6 | The change at each latitude (after Bracegirdle et al., 2024, Fig. 2), every model as a line, with the sea-ice edge (JJA) |
| S1 | Fig. 1 in summer (DJF) |
| S2, S3 | Fig. 2 and 3 for every model on its own (historical) |
| S4–S7 | How many models have a significant change in the mean, and in the width (DJF, JJA) |
| S8, S9 | S/N of every model (DJF, JJA) |
| S10 | The change at every 5th percentile, every model (DJF, JJA) |
| S11 | Additivity: historical against the sum of the single forcings, every model (JJA) |
| S12 | Fig. 6 in summer (DJF) |

**Putting the models together.** A multi-model panel is the median across models. A change is **robust** where at least two-thirds of the models (`ROBUST`, `config.SIGNIFICANT_THRESHOLD`) have a significant change of the same sign (the IPCC AR6 approach); dots mark where it is not. The tests are those of notebook 03: the member-block test for the mean, the hist-nat bootstrap for the width (of the internal variability); single quantiles have no test.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import config, paths, response_change as rc, sea_ice, significance as sig, storage, zonal
from extant.logger import setup_logging
from extant.plots import forced_response as fr, zonal as zp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units
WINDOW = config.WINDOW

#(t): The forcings, in the order of every figure; the seasons; and the winter and summer of the main figures
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
WINTER, SUMMER = 'JJA', 'DJF'

#(t): Significance level of every test; |S/N| counted as emerged; the fraction of models that makes a change robust
ALPHA = 0.05
SN_THRESHOLD = 2
ROBUST = config.SIGNIFICANT_THRESHOLD

#(t): The test of the mean, as in notebook 04 section 5: 'significant' is the member-block test
MEAN_TEST = 'significant'

#(t): Print style; the width of every figure, a two-column page (183 mm, which each figure's panels fill); and the
#(t): most a map figure can be tall (229 mm: its panels shrink to fit, leaving room for the caption)
STYLE = fr.PAPER
WIDTH = fr.PAGE_WIDTH
HEIGHT = fr.PAGE_HEIGHT


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

In [ ]:
#(t): The same colour scales as notebook 04
scales = fr.colour_scales(VARIABLE, alpha=ALPHA, sn_threshold=SN_THRESHOLD)

## Open the saved results

Run **one** of the next two sections. Both leave `RUN`, which says which saved results to open, and `PAPER_DIR`, where the figures go.

### Sample data

In [ ]:
RUN = 'sample'
PAPER_DIR = paths.figure('paper_sample', VARIABLE)
PAPER_DIR

### Full data (JASMIN)

In [ ]:
RUN = 'full'
PAPER_DIR = paths.figure('paper', VARIABLE)
PAPER_DIR

### The saved results

In [ ]:
final_mean_ds = storage.open_dataset(result_path('final_mean'), load=True)
mean_block_ds = storage.open_dataset(result_path('mean_block'), load=True)
lesfmip_seasonal_sn_ds = storage.open_dataset(result_path('sn'), load=True)
q_histnat_base_ds = storage.open_dataset(result_path('q_histnat_base'), load=True)
final_q_da = storage.open_dataarray(result_path('final_quantiles'), load=True)
quantile_curve_da = storage.open_dataarray(result_path('quantile_curve'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
tails_ds = storage.open_dataset(result_path('tails'), load=True)
own_changes_da = storage.open_dataarray(result_path('own_changes'), load=True)
zonal_mean_ds = storage.open_dataset(result_path('zonal_mean_test'), load=True)
zonal_width_ds = storage.open_dataset(result_path('zonal_width_test'), load=True)

In [ ]:
#(t): The models: every count runs up to this many
N_MODELS = mean_block_ds.sizes['model']
list(mean_block_ds['model'].values)

### Sea ice

For the ice edge in Figs 6 and S12 (notebook 01, section 5). Run **one** of the next two sections.

#### Without sea ice

In [ ]:
ice_edges_da = None

#### With sea ice (JASMIN)

In [ ]:
ice_extents_tree = storage.open_tree(paths.result('extent', 'siconc', 'full'), load=True)
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)

## The fields

As in notebook 04, section 5: each part of the distribution in hist-nat and in every experiment, the change in each, and which changes are significant in each model.

In [ ]:
#(t): S/N in the centre of the final WINDOW years, for the 'emerged' and 'robust' mean tests
sn_final_da = lesfmip_seasonal_sn_ds[VARIABLE].isel(year=config.LAST_FULL_YEAR)
sn_final_da

In [ ]:
#(t): Every change and its test, on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(mean_block_ds['mean_change'], mean_block_ds['pvalue'], sn_final_da, qrange_change_ds,
                              tails_ds, alpha=ALPHA, sn_threshold=SN_THRESHOLD)
change_ds

In [ ]:
#(t): hist-nat's distribution: Q05 and Q95 over its whole record, the mean over the final WINDOW years, and the
#(t): width of its internal variability
hist_nat_fields_ds = xr.Dataset({
    'Q05': q_histnat_base_ds[VARIABLE].sel(quantile=0.05, drop=True),
    'Mean': final_mean_ds[VARIABLE].sel(experiment='hist-nat', drop=True),
    'Q95': q_histnat_base_ds[VARIABLE].sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['hist_nat_qrange'],
})
hist_nat_fields_ds

In [ ]:
#(t): Every experiment's, over its final WINDOW years
experiment_fields_ds = xr.Dataset({
    'Q05': final_q_da.sel(quantile=0.05, drop=True),
    'Mean': final_mean_ds[VARIABLE].drop_sel(experiment='hist-nat'),
    'Q95': final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['experiment_qrange'],
})
experiment_fields_ds

In [ ]:
#(t): The change in each
change_fields_ds = experiment_fields_ds - hist_nat_fields_ds
change_fields_ds

In [ ]:
#(t): Which changes are significant in each model: the mean by MEAN_TEST, the width by the hist-nat bootstrap
change_significant_ds = xr.Dataset({'Mean': change_ds[f'mean_{MEAN_TEST}'], 'Q95 − Q05': change_ds['width_significant']})
change_significant_ds

In [ ]:
#(t): Not significant, where there is data
change_not_significant_ds = ~change_significant_ds & change_fields_ds[list(change_significant_ds)].notnull()

In [ ]:
#(t): The sign of each model's significant changes: +1 up, -1 down, 0 not significant
mean_sign_da = sig.significant_sign(change_fields_ds['Mean'], change_significant_ds['Mean'])
width_sign_da = sig.significant_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): Each model's joint class (notebook 04, section 5)
model_joint_class_da = rc.joint_class(mean_sign_da, width_sign_da)
model_joint_class_da

## The models together

The median across models of every field, and where the models agree: the sign of a change where at least `ROBUST` of the models with data have a significant change of that sign, 0 where they do not.

In [ ]:
#(t): The multi-model median of every field: hist-nat
median_hist_nat_fields_ds = hist_nat_fields_ds.median('model')
median_hist_nat_fields_ds

In [ ]:
#(t): every experiment
median_experiment_fields_ds = experiment_fields_ds.median('model')

In [ ]:
#(t): and the change
median_change_fields_ds = change_fields_ds.median('model')

In [ ]:
#(t): Where the models agree on a significant change in the mean
mean_robust_sign_da = sig.robust_sign(change_fields_ds['Mean'], change_significant_ds['Mean'], threshold=ROBUST)
mean_robust_sign_da

In [ ]:
#(t): and in the width
width_robust_sign_da = sig.robust_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'],
                                       threshold=ROBUST)

In [ ]:
#(t): Not robust, where there is data: the dots
not_robust_ds = xr.Dataset({'Mean': mean_robust_sign_da == 0, 'Q95 − Q05': width_robust_sign_da == 0})

In [ ]:
#(t): The joint class of the robust changes
robust_joint_class_da = rc.joint_class(mean_robust_sign_da, width_robust_sign_da)
robust_joint_class_da

In [ ]:
#(t): A check on the maps: 1 where the models agree on a change, 0 where not (NaN without data)
robust_ds = xr.Dataset({'mean': np.abs(mean_robust_sign_da), 'width': np.abs(width_robust_sign_da)})

In [ ]:
#(t): How much of the area south of 60°S each forcing robustly changes, in each season (%)
robust_area_ds = 100 * sig.area_mean(robust_ds.sel(lat=slice(None, -60)))
robust_area_ds.to_dataframe().round(0)

## Check the colour ranges

How much of each field falls beyond the ends of its colour scale, where the maps show only the end colour (and the colour bar an arrow). A few percent is fine. Much more means the range in `fr.RANGES` is too narrow for these data (set it there, and every notebook follows), or a model has gone wrong (notebook 04's *Check the results*). Every model's field is checked, not just the median, so the supplementary figures are covered too.

In [ ]:
#(t): Each field the figures draw, with the scale it is drawn with
range_checks = {
    'hist-nat: Q05, mean, Q95': (hist_nat_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.temperature),
    'experiments: Q05, mean, Q95': (experiment_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.temperature),
    'Q95 - Q05': (experiment_fields_ds['Q95 − Q05'], scales.spread),
    'change in Q05, mean, Q95': (change_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.change),
    'change in Q95 - Q05': (change_fields_ds['Q95 − Q05'], scales.spread_change),
    'S/N': (sn_final_da, scales.sn),
}

In [ ]:
#(t): The share beyond each end (%)
saturation_df = 100 * pd.DataFrame({name: fr.saturation(da, scale) for name, (da, scale) in range_checks.items()}).T
saturation_df.round(1)

## Main figures

**Figure 1. Historical change in the distribution of winter (JJA) near-surface air temperature.** Multi-model median of (a–d) hist-nat (Q05 and Q95 over its whole record, the mean over 1993–2013), (e–h) historical over 1993–2013, and (i–l) the change between them: the 5th percentile, the mean, the 95th percentile, and the width Q95 − Q05 of the internal variability (each member minus its ensemble's forced response). Dots: fewer than two-thirds of the models have a significant change of one sign (the mean: member-block permutation test; the width: hist-nat bootstrap; p < 0.05). After Bracegirdle et al. (2024), Fig. 1.

In [ ]:
panels = fr.block_grid(
    [fr.Block({'hist-nat': median_hist_nat_fields_ds.sel(season=WINTER),
               'historical': median_experiment_fields_ds.sel(experiment='historical', season=WINTER)},
              scales=[scales.temperature, scales.temperature, scales.temperature, scales.spread]),
     fr.Block({'Change': median_change_fields_ds.sel(experiment='historical', season=WINTER)},
              scales=[scales.change, scales.change, scales.change, scales.spread_change],
              not_significant={'Change': not_robust_ds.sel(experiment='historical', season=WINTER)})],
    width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label='Not robust (fewer than 2/3 of the models significant)',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'fig1_distribution_historical_{WINTER}')

**Figure 2. How each forcing changes the distribution of winter (JJA) temperature.** Multi-model median change, 1993–2013 against hist-nat, of the 5th percentile, the mean, the 95th percentile and the width (Q95 − Q05) of the internal variability, for each single forcing (rows) and all forcings together (historical). Dots as in Fig. 1.

In [ ]:
#(t): The changes in WINTER, every forcing
winter_changes_ds = median_change_fields_ds.sel(experiment=EXPERIMENTS, season=WINTER)
winter_changes_ds

In [ ]:
#(t): and where they are not robust
winter_not_robust_ds = not_robust_ds.sel(experiment=EXPERIMENTS, season=WINTER)

In [ ]:
panels = fr.block_grid(
    [fr.Block({experiment: winter_changes_ds.sel(experiment=experiment) for experiment in EXPERIMENTS},
              scales=[scales.change, scales.change, scales.change, scales.spread_change],
              not_significant={experiment: winter_not_robust_ds.sel(experiment=experiment) for experiment in EXPERIMENTS})],
    column_titles=['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)'],
    width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label='Not robust (fewer than 2/3 of the models significant)',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'fig2_forcings_{WINTER}')

**Figure 3. As Fig. 2, in summer (DJF).**

In [ ]:
#(t): The changes in SUMMER, every forcing
summer_changes_ds = median_change_fields_ds.sel(experiment=EXPERIMENTS, season=SUMMER)
summer_changes_ds

In [ ]:
#(t): and where they are not robust
summer_not_robust_ds = not_robust_ds.sel(experiment=EXPERIMENTS, season=SUMMER)

In [ ]:
panels = fr.block_grid(
    [fr.Block({experiment: summer_changes_ds.sel(experiment=experiment) for experiment in EXPERIMENTS},
              scales=[scales.change, scales.change, scales.change, scales.spread_change],
              not_significant={experiment: summer_not_robust_ds.sel(experiment=experiment) for experiment in EXPERIMENTS})],
    column_titles=['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)'],
    width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label='Not robust (fewer than 2/3 of the models significant)',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'fig3_forcings_{SUMMER}')

**Figure 4. Where the mean and the variability have changed together.** For each forcing (columns) in summer (DJF) and winter (JJA): the robust change in the mean (rows of the key: lower, none, higher) combined with the robust change in the width of the internal variability (columns of the key: narrower, none, wider). A change is robust where at least two-thirds of the models have a significant change of that sign (tests as in Fig. 1).

In [ ]:
#(t): The robust joint class, every forcing, in summer and winter
seasons_class_da = robust_joint_class_da.sel(experiment=EXPERIMENTS, season=[SUMMER, WINTER])
seasons_class_da

In [ ]:
panels = fr.map_grid(
    seasons_class_da, scales.joint, row_dim='season', col_dim='experiment',
    width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
)

In [ ]:
fr.save(panels, PAPER_DIR, 'fig4_joint_change')

**Figure 5. The change in the mean and in the width, averaged south of 60°S.** Each model's area-mean change (markers; colour: forcing) in each season, 1993–2013 against hist-nat; bars: the multi-model median. Top: the mean; bottom: the width Q95 − Q05 of the internal variability.

In [ ]:
#(t): Area means south of 60°S of every model's changes
change_regional_ds = rc.regional_mean(change_ds.sel(experiment=EXPERIMENTS), lat_max=-60, mean_test=MEAN_TEST)
change_regional_ds

In [ ]:
panels = fr.forcing_dots(
    change_regional_ds,
    {'mean_change': f'Δ mean ({UNITS})', 'width_change': f'Δ(Q95 − Q05) ({UNITS})'},
    columns=SEASONS, experiments=EXPERIMENTS, style=STYLE, panel_size=(1.45, 1.5), width=WIDTH,
)

In [ ]:
fr.save(panels, PAPER_DIR, 'fig5_area_means')

**Figure 6. The change at each latitude in winter (JJA).** Zonal-mean change, 1993–2013 against hist-nat, for each forcing (rows) and every model (lines): the mean; the low extremes Δ(Q05 − Q50) and the high extremes Δ(Q95 − Q50) of the internal variability, each measured from the median; and the width Δ(Q95 − Q05), the high minus the low. Dots: significant zonal-mean change in that model (the mean: member-block test; the width: hist-nat bootstrap; p < 0.05). Dashed: the sea-ice edge (equivalent latitude) of each model in 1993–2013. After Bracegirdle et al. (2024), Fig. 2.

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)
zonal_ds

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, WINTER, rows='experiment', lines='model', row_values=EXPERIMENTS,
                                 edges=ice_edges_da, units=UNITS, title='', panel_size=(1.55, 1.15),
                                 rc_params=STYLE.rc, width=WIDTH)

In [ ]:
fr.save(fig, PAPER_DIR, f'fig6_zonal_{WINTER}')

## Supplementary figures

**Figure S1. As Fig. 1, in summer (DJF).**

In [ ]:
panels = fr.block_grid(
    [fr.Block({'hist-nat': median_hist_nat_fields_ds.sel(season=SUMMER),
               'historical': median_experiment_fields_ds.sel(experiment='historical', season=SUMMER)},
              scales=[scales.temperature, scales.temperature, scales.temperature, scales.spread]),
     fr.Block({'Change': median_change_fields_ds.sel(experiment='historical', season=SUMMER)},
              scales=[scales.change, scales.change, scales.change, scales.spread_change],
              not_significant={'Change': not_robust_ds.sel(experiment='historical', season=SUMMER)})],
    width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label='Not robust (fewer than 2/3 of the models significant)',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS1_distribution_historical_{SUMMER}')

**Figure S2. Every model's historical change in the distribution of winter (JJA) temperature.** As the bottom row of Fig. 1, for each model (rows), with each model's joint class (as in Fig. 4, from that model's own tests). Dots: not significant in that model (p ≥ 0.05).

In [ ]:
#(t): Every model, historical, WINTER: the changes
winter_model_changes_ds = change_fields_ds.sel(experiment='historical', season=WINTER)
winter_model_changes_ds

In [ ]:
#(t): where they are not significant
winter_model_not_significant_ds = change_not_significant_ds.sel(experiment='historical', season=WINTER)

In [ ]:
#(t): and the joint class
winter_model_class_da = model_joint_class_da.sel(experiment='historical', season=WINTER)

In [ ]:
panels = fr.field_grid(
    [fr.Column(winter_model_changes_ds['Q05'], scales.change, 'ΔQ05'),
     fr.Column(winter_model_changes_ds['Mean'], scales.change, 'Δ mean',
               not_significant=winter_model_not_significant_ds['Mean']),
     fr.Column(winter_model_changes_ds['Q95'], scales.change, 'ΔQ95'),
     fr.Column(winter_model_changes_ds['Q95 − Q05'], scales.spread_change, 'Δ(Q95 − Q05)',
               not_significant=winter_model_not_significant_ds['Q95 − Q05']),
     fr.Column(winter_model_class_da, scales.joint, 'What changed')],
    row_dim='model', width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS2_models_historical_{WINTER}')

**Figure S3. As Fig. S2, in summer (DJF).**

In [ ]:
#(t): Every model, historical, SUMMER: the changes
summer_model_changes_ds = change_fields_ds.sel(experiment='historical', season=SUMMER)
summer_model_changes_ds

In [ ]:
#(t): where they are not significant
summer_model_not_significant_ds = change_not_significant_ds.sel(experiment='historical', season=SUMMER)

In [ ]:
#(t): and the joint class
summer_model_class_da = model_joint_class_da.sel(experiment='historical', season=SUMMER)

In [ ]:
panels = fr.field_grid(
    [fr.Column(summer_model_changes_ds['Q05'], scales.change, 'ΔQ05'),
     fr.Column(summer_model_changes_ds['Mean'], scales.change, 'Δ mean',
               not_significant=summer_model_not_significant_ds['Mean']),
     fr.Column(summer_model_changes_ds['Q95'], scales.change, 'ΔQ95'),
     fr.Column(summer_model_changes_ds['Q95 − Q05'], scales.spread_change, 'Δ(Q95 − Q05)',
               not_significant=summer_model_not_significant_ds['Q95 − Q05']),
     fr.Column(summer_model_class_da, scales.joint, 'What changed')],
    row_dim='model', width=WIDTH, height=HEIGHT, style=STYLE, tag=True,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS3_models_historical_{SUMMER}')

**Figures S4–S7. How many models have a significant change.** At each grid point, the number of models with a significant increase (top rows, red) and decrease (bottom rows, blue) in the mean (S4: DJF, S5: JJA) and in the width Q95 − Q05 (S6: DJF, S7: JJA), for each forcing. Tests as in Fig. 1.

In [ ]:
#(t): The counts: the mean
mean_counts_da = sig.significant_counts(change_fields_ds['Mean'], change_significant_ds['Mean'])
mean_counts_da

In [ ]:
#(t): and the width
width_counts_da = sig.significant_counts(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
panels = fr.count_grid(mean_counts_da.sel(experiment=EXPERIMENTS, season=SUMMER), N_MODELS, title=None,
                       label='Models with a significant decrease (blue) or increase (red) in the mean',
                       row_labels={'increase': 'Increase', 'decrease': 'Decrease'},
                       width=WIDTH, height=HEIGHT, style=STYLE, tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS4_mean_counts_{SUMMER}')

In [ ]:
panels = fr.count_grid(mean_counts_da.sel(experiment=EXPERIMENTS, season=WINTER), N_MODELS, title=None,
                       label='Models with a significant decrease (blue) or increase (red) in the mean',
                       row_labels={'increase': 'Increase', 'decrease': 'Decrease'},
                       width=WIDTH, height=HEIGHT, style=STYLE, tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS5_mean_counts_{WINTER}')

In [ ]:
panels = fr.count_grid(width_counts_da.sel(experiment=EXPERIMENTS, season=SUMMER), N_MODELS, title=None,
                       label='Models significantly narrower (blue) or wider (red)',
                       row_labels={'increase': 'Wider', 'decrease': 'Narrower'},
                       width=WIDTH, height=HEIGHT, style=STYLE, tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS6_width_counts_{SUMMER}')

In [ ]:
panels = fr.count_grid(width_counts_da.sel(experiment=EXPERIMENTS, season=WINTER), N_MODELS, title=None,
                       label='Models significantly narrower (blue) or wider (red)',
                       row_labels={'increase': 'Wider', 'decrease': 'Narrower'},
                       width=WIDTH, height=HEIGHT, style=STYLE, tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS7_width_counts_{WINTER}')

**Figures S8, S9. Signal to noise.** S/N in 2003 (the centre of 1993–2013) for every model (rows) and forcing (columns), in DJF (S8) and JJA (S9): the forced response minus hist-nat's, over the pooled year-to-year standard deviation. White: |S/N| < 2, not emerged.

In [ ]:
panels = fr.map_grid(sn_final_da.sel(experiment=EXPERIMENTS, season=SUMMER), scales.sn, width=WIDTH, height=HEIGHT, style=STYLE,
                     tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS8_sn_{SUMMER}')

In [ ]:
panels = fr.map_grid(sn_final_da.sel(experiment=EXPERIMENTS, season=WINTER), scales.sn, width=WIDTH, height=HEIGHT, style=STYLE,
                     tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS9_sn_{WINTER}')

**Figure S10. The change at every percentile.** Area mean (south of 40°S) of the change in every 5th percentile, 1993–2013 against hist-nat's whole record, every model (lines), in DJF (top) and JJA (bottom). A flat line is a pure shift; a rising line, a widening distribution.

In [ ]:
panels = fr.quantile_curves(quantile_curve_da.sel(experiment=EXPERIMENTS, season=SUMMER), title=f'{SUMMER}', units=UNITS,
                            style=STYLE, panel_size=(1.45, 1.4), width=WIDTH)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS10a_quantile_curves_{SUMMER}')

In [ ]:
panels = fr.quantile_curves(quantile_curve_da.sel(experiment=EXPERIMENTS, season=WINTER), title=f'{WINTER}', units=UNITS,
                            style=STYLE, panel_size=(1.45, 1.4), width=WIDTH)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS10b_quantile_curves_{WINTER}')

**Figure S11. Additivity.** For each model (rows), winter (JJA): the sum of the four single-forcing changes (hist-nat, hist-GHG, hist-aer, hist-totalO3), the historical change, and the residual (historical minus the sum), each experiment's final 21 years against its own 1850–1900.

In [ ]:
#(t): Every term: each part, their sum, historical and the residual
additivity_ds = rc.additivity(own_changes_da)
additivity_ds

In [ ]:
#(t): The sum, historical and the residual, every model, WINTER
winter_additivity_da = additivity_ds['response'].sel(season=WINTER, term=['sum of parts', 'historical', 'residual'])
winter_additivity_da

In [ ]:
panels = fr.map_grid(winter_additivity_da, scales.change, row_dim='model', col_dim='term', width=WIDTH, height=HEIGHT,
                     style=STYLE, tag=True)

In [ ]:
fr.save(panels, PAPER_DIR, f'figS11_additivity_{WINTER}')

**Figure S12. As Fig. 6, in summer (DJF).**

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, SUMMER, rows='experiment', lines='model', row_values=EXPERIMENTS,
                                 edges=ice_edges_da, units=UNITS, title='', panel_size=(1.55, 1.15),
                                 rc_params=STYLE.rc, width=WIDTH)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS12_zonal_{SUMMER}')

## What was saved

In [ ]:
pd.Series(sorted(path.name for path in PAPER_DIR.glob('*.pdf')), name='saved in ' + str(PAPER_DIR))